In [0]:
# ============================================================
# GOLD LAYER - DIM_CUSTOMER - CORRECTED VERSION
# ============================================================

from pyspark.sql import functions as F
from datetime import datetime

# ============================================================
# GET ENVIRONMENT FROM ADF
# ============================================================

try:
    env = dbutils.widgets.get("environment")
    print(f"📌 Environment from ADF: {env}")
except:
    env = "DEV"
    print(f"📌 Using default: {env}")

# ============================================================
# STORAGE ACCOUNT
# ============================================================

storage_account_name = "capstonestorage01"

# ============================================================
# CONTAINER NAMES BASED ON ENVIRONMENT
# ============================================================

if env == 'DEV':
    silver_container = 'silver-dev'
    gold_container = 'gold-dev'
elif env == 'TEST':
    silver_container = 'silver-test'
    gold_container = 'gold-test'
else:  # PROD
    silver_container = 'silver'
    gold_container = 'gold'

# ============================================================
# BUILD PATHS
# ============================================================

SILVER = f"abfss://{silver_container}@{storage_account_name}.dfs.core.windows.net"
GOLD = f"abfss://{gold_container}@{storage_account_name}.dfs.core.windows.net"

print(f"\n{'='*55}")
print(f"🏭 ENVIRONMENT: {env}")
print(f"{'='*55}")
print(f"📁 SILVER Container: {silver_container}")
print(f"📁 GOLD Container: {gold_container}")
print(f"📍 SILVER Path: {SILVER}")
print(f"📍 GOLD Path: {GOLD}")
print(f"{'='*55}\n")

# ============================================================
# LOAD FROM SILVER
# ============================================================

df_customer = spark.read.format("delta").load(f"{SILVER}/customer_master")
print(f"📖 Loaded customer records: {df_customer.count():,}")

# ============================================================
# TRANSFORM - CREATE DIM_CUSTOMER
# ============================================================

dim_customer = df_customer.select(
    F.col("customer_id"),
    F.col("age"),
    F.col("gender"),
    F.col("city"),
    F.col("employment_type"),
    F.col("annual_income"),
    F.col("risk_segment"),
    
    # Income band
    F.when(F.col("annual_income") < 500000, F.lit("Low Income"))
     .when(F.col("annual_income") < 1000000, F.lit("Mid Income"))
     .otherwise(F.lit("High Income"))
     .alias("income_band"),
    
    # Age group
    F.when(F.col("age") < 30, F.lit("18-29"))
     .when(F.col("age") < 45, F.lit("30-44"))
     .when(F.col("age") < 60, F.lit("45-59"))
     .otherwise(F.lit("60+"))
     .alias("age_group"),
    
    F.current_timestamp().alias("gold_created_at")
)

print(f"🔄 Transformed rows: {dim_customer.count():,}")

# ============================================================
# WRITE TO GOLD
# ============================================================

dim_customer.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .save(f"{GOLD}/dim_customer")

# ============================================================
# SUMMARY
# ============================================================

print(f"\n{'='*55}")
print(f"✅ dim_customer : {dim_customer.count():,} rows")
print(f"📁 Written to: {GOLD}/dim_customer")
print(f"🏭 Environment: {env}")
print(f"{'='*55}")